# 🚀 Python İleri Seviye Konular — Bölüm 2

Bu bölümde **Abstract Method**, **Type Annotations**, **Overloading**, **Overriding**, **Final** ve **Decorator Combining** konularını ele alacağız. Konular birbirini tamamlayan bir yapıda ilerliyor — sırayla okumak en verimli yol.

---

## 🔹 1. Abstract Method (Soyut Metot)

### Sorun: "Sözleşmesiz" Kalıtım

Bir ödeme sistemi geliştirdiğini düşün. Kredi kartı, PayPal, kripto — hepsi farklı ödeme yöntemleri ama hepsinin ortak bir arayüzü olması gerekiyor: `ode()` ve `iade_et()`.

Bunu sıradan kalıtımla yaparsan:
```python
class OdemeSistemi:
    def ode(self, miktar):
        pass  # alt sınıf yazmayı unutursa? sessizce geçer
```

Alt sınıf `ode()` metodunu implement etmeyi unutsa bile Python **hiçbir hata vermez.** Sorun ancak çalışma zamanında, muhtemelen çok geç bir noktada ortaya çıkar.

### Çözüm: Abstract Class
```python
from abc import ABC, abstractmethod

class OdemeSistemi(ABC):

    @abstractmethod
    def ode(self, miktar: float) -> bool:
        pass

    @abstractmethod
    def iade_et(self, miktar: float) -> bool:
        pass
```
```python
class KrediKarti(OdemeSistemi):
    def ode(self, miktar):
        print(f"💳 Kredi kartıyla {miktar}₺ ödendi")
        return True

    def iade_et(self, miktar):
        print(f"💳 {miktar}₺ iade edildi")
        return True
```
```python
class PayPal(OdemeSistemi):
    def ode(self, miktar):
        print(f"🅿️ PayPal ile {miktar}₺ ödendi")
        return True

    # iade_et() yazılmadı!
```
```python
kk = KrediKarti()    # ✅ çalışır
pp = PayPal()        # ❌ TypeError: PayPal'ın iade_et() metodu yok
```

> `ABC`'den türeyen bir class, **tüm abstract metodları implement etmediği sürece** instance oluşturulamaz.
> Hata çalışma zamanında değil, obje oluşturma anında verilir.

### 💡 Abstract Class'ın 3 Kuralı
```
1. ABC'den miras alınır
2. @abstractmethod ile işaretlenen metodlar alt sınıfta ZORUNLU implement edilmeli
3. ABC'nin kendisinden doğrudan obje oluşturulamaz
```
```python
sistem = OdemeSistemi()  # ❌ TypeError: soyut class'tan obje üretilemez
```

### 🎯 Ne Zaman Kullanılır?

Birden fazla sınıfın **aynı arayüzü paylaşması gerektiğinde** ama her birinin onu **farklı uygulaması** gerektiğinde. Örnek: veritabanı bağlantıları, dosya okuyucular, bildirim sistemleri.

---

## 🔹 2. Type Annotations (Tip İpuçları)

Abstract method örneklerinde `miktar: float` ve `-> bool` ifadelerini fark ettin mi? Bunlar **type annotation** — Python'un opsiyonel tip sistemi.
```python
# Annotation'sız
def kullanici_bul(id):
    ...

# Annotation'lı
def kullanici_bul(id: int) -> dict:
    ...
```

### ⚠️ En Kritik Nokta
```python
def yas_iki_katina_al(yas: int) -> int:
    return yas * 2

print(yas_iki_katina_al("yirmi"))  # ✅ hata vermez, "yirmiyirmiyirmiyirmi" döner!
```

> Type annotation'lar **runtime'da hiçbir kontrol yapmaz.**
> Python bu kuralları çalışma zamanında **zorlamaz.**
> Amaç: kodun okunabilirliğini artırmak, IDE ve statik analiz araçlarının
> (mypy, Pyright) hataları **önceden yakalamasını** sağlamak.

### Yaygın Annotation Örnekleri
```python
from typing import Union, Optional, List

def isle(veri: Union[int, str]) -> str:       # int ya da str kabul eder
    return str(veri)

def ara(isim: Optional[str] = None) -> list:  # str ya da None
    ...

def topla(sayilar: List[int]) -> int:         # int listesi
    return sum(sayilar)

# Python 3.10+ kısa sözdizimi
def isle(veri: int | str) -> str:
    ...
```

---

## 🔹 3. Overloading (Aşırı Yükleme)

### Python'da "Gerçek" Overloading Yoktur

Java veya C++'ta aynı isimli, farklı parametreli birden fazla metot tanımlanabilir. Python'da son tanımlanan **öncekini siler:**
```python
class Yazici:

    def yaz(self, metin: str):
        print(metin)

    def yaz(self, metin: str, renk: str):  # bu, üsttekini ezip geçer
        print(f"[{renk}] {metin}")

y = Yazici()
y.yaz("Merhaba")  # ❌ TypeError: 'renk' argümanı eksik
```

### Yol 1 — Optional Parametre (Pratik Çözüm)
```python
class Yazici:

    def yaz(self, metin: str, renk: str | None = None) -> None:
        if renk is None:
            print(metin)
        else:
            print(f"[{renk}] {metin}")
```
```python
y = Yazici()
y.yaz("Merhaba")              # ✅ Merhaba
y.yaz("Hata!", "kırmızı")    # ✅ [kırmızı] Hata!
```

### Yol 2 — `@overload` ile IDE Desteği

`@overload` runtime'da **hiçbir şey değiştirmez.** IDE ve type checker'ların her çağrı için doğru tipi görmesini sağlar:
```python
from typing import overload, Union

class DonusturucU:

    @overload
    def donustur(self, veri: int) -> str: ...

    @overload
    def donustur(self, veri: list) -> str: ...

    def donustur(self, veri: Union[int, list]) -> str:
        if isinstance(veri, int):
            return f"Sayı: {veri}"
        elif isinstance(veri, list):
            return f"Liste: {', '.join(map(str, veri))}"
        raise ValueError("Desteklenmeyen tip")
```
```python
d = DonusturucU()
print(d.donustur(42))           # Sayı: 42
print(d.donustur([1, 2, 3]))    # Liste: 1, 2, 3
```

> `@overload` ile işaretlenen metodların gövdesi `...` olmalı — bunlar sadece
> tip bilgisi için var. Asıl iş **son tanımlanan gerçek metotta** yapılır.

---

## 🔹 4. Overriding (Ezme / Geçersiz Kılma)

Alt sınıfın, üst sınıftan gelen bir metodu **kendi ihtiyacına göre yeniden yazmasına** denir.
```python
class Bildirim:

    def gonder(self, mesaj: str) -> None:
        print(f"📨 Bildirim gönderildi: {mesaj}")


class EmailBildirim(Bildirim):

    def __init__(self, alici: str):
        self.alici = alici

    def gonder(self, mesaj: str) -> None:  # override
        print(f"📧 {self.alici} adresine e-posta: {mesaj}")


class SMSBildirim(Bildirim):

    def __init__(self, telefon: str):
        self.telefon = telefon

    def gonder(self, mesaj: str) -> None:  # override
        print(f"📱 {self.telefon} numarasına SMS: {mesaj}")
```
```python
bildirimler = [
    EmailBildirim("ali@mail.com"),
    SMSBildirim("+90 555 000 00 00"),
    Bildirim()
]

for b in bildirimler:
    b.gonder("Siparişiniz kargoya verildi")
```
```
📧 ali@mail.com adresine e-posta: Siparişiniz kargoya verildi
📱 +90 555 000 00 00 numarasına SMS: Siparişiniz kargoya verildi
📨 Bildirim gönderildi: Siparişiniz kargoya verildi
```

### `@override` Decorator — Güvenli Ezme
```python
from typing import override

class EmailBildirim(Bildirim):

    @override  # IDE'ye "bu bir override" diye bildiriyoruz
    def gonder(self, mesaj: str) -> None:
        print(f"📧 E-posta: {mesaj}")
```

`@override` ne işe yarar? Üst sınıfta metot adı değiştirilirse veya silinirse IDE/type checker **hemen uyarır.** Runtime'da etkisi yoktur.
```python
# Üst sınıfta 'gonder' → 'ilet' olarak değiştirildiğinde:
# @override olan EmailBildirim.gonder → mypy hata verir ✅
# @override olmayan → sessizce yeni bir metot gibi davranır ❌
```

---

## 🔹 5. Final — "Bu Değişmesin"

Bazen bir metodun alt sınıflar tarafından **override edilmesini istemezsin.** Lisans doğrulama mantığı, kritik iş kuralları... `@final` tam bu iş için:
```python
from typing import final

class OdemeIslemcisi:

    def odeme_yap(self, miktar: float) -> bool:
        # Alt sınıflar bunu özelleştirebilir
        return self._bankaya_gonder(miktar)

    @final
    def komisyon_hesapla(self, miktar: float) -> float:
        # Bu kural kesinlikle değiştirilmemeli
        return miktar * 0.029 + 0.30


class OzelIslemci(OdemeIslemcisi):

    def odeme_yap(self, miktar: float) -> bool:  # ✅ override edilebilir
        print("Özel ödeme akışı")
        return True

    def komisyon_hesapla(self, miktar: float) -> float:  # ⚠️ mypy uyarı verir
        return 0  # komisyonu sıfırlamaya çalışıyoruz ama olmamalı
```

> ⚠️ `@final` runtime'da **engellemez** — Python bunu çalıştırmaya devam eder.
> Koruma IDE ve type checker katmanında çalışır (mypy, Pyright vb.).
> Ekiple çalışıyorsan bu yeterlidir; CI/CD pipeline'ına mypy eklemek koruyu tamamlar.

### `@final` Class Seviyesinde de Kullanılabilir
```python
@final
class Sabit:
    PI = 3.14159

class YeniSabit(Sabit):  # ⚠️ mypy: 'Sabit' final olarak işaretlenmiş
    pass
```

---

## 🔹 6. Bonus — Decorator Combining (Dekoratör Zincirleme)

Birden fazla decorator aynı fonksiyona uygulanabilir. Sıralama önemlidir:
```python
def buyuk_harf(func):
    def wrapper(metin: str):
        return func(metin).upper()
    return wrapper

def ünlem_ekle(func):
    def wrapper(metin: str):
        return func(metin) + "!!!"
    return wrapper

@buyuk_harf
@ünlem_ekle
def selamla(isim: str) -> str:
    return f"Merhaba {isim}"

print(selamla("Ahmet"))  # MERHABA AHMET!!!
```

### Uygulama Sırası
```
@buyuk_harf
@ünlem_ekle
def selamla: ...
```

Bu şunu demektir:
```python
selamla = buyuk_harf(ünlem_ekle(selamla))
```

Yani **içten dışa** uygulanır:
```
selamla("Ahmet")
  → ünlem_ekle: "Merhaba Ahmet!!!"
    → buyuk_harf: "MERHABA AHMET!!!"
```

> Decorator sırası değişirse sonuç da değişir — dikkatli ol.

### Gerçek Hayat Örneği
```python
def giris_gerekli(func):
    def wrapper(kullanici, *args, **kwargs):
        if not kullanici.get("giris_yapti"):
            raise PermissionError("Giriş yapmanız gerekiyor")
        return func(kullanici, *args, **kwargs)
    return wrapper

def admin_gerekli(func):
    def wrapper(kullanici, *args, **kwargs):
        if not kullanici.get("admin"):
            raise PermissionError("Admin yetkisi gerekiyor")
        return func(kullanici, *args, **kwargs)
    return wrapper

@giris_gerekli
@admin_gerekli
def kullanici_sil(kullanici, hedef_id: int):
    print(f"🗑️ Kullanıcı #{hedef_id} silindi")
```
```python
admin = {"giris_yapti": True, "admin": True}
misafir = {"giris_yapti": False, "admin": False}

kullanici_sil(admin, 42)     # ✅ Kullanıcı #42 silindi
kullanici_sil(misafir, 42)   # ❌ PermissionError: Giriş yapmanız gerekiyor
```

---

## 📊 Bölüm Özeti

| Kavram | Modül | Runtime Etkisi | Temel Amacı |
|---|---|:---:|---|
| `ABC` + `@abstractmethod` | `abc` | ✅ Var | Alt sınıfı implement etmeye **zorlar** |
| Type Annotation | `typing` | ❌ Yok | Okunabilirlik, IDE desteği |
| `@overload` | `typing` | ❌ Yok | Farklı tipler için tip ipucu |
| `@override` | `typing` | ❌ Yok | Override hatasını **önceden yakalar** |
| `@final` | `typing` | ❌ Yok | Override/miras'ı **kısıtlar** (IDE düzeyinde) |
| Decorator Combining | — | ✅ Var | Fonksiyona birden fazla sorumluluk ekler |

---

## 🎯 Akılda Kalması Gereken 5 Kural

1. **Abstract class** → "Bu metodları implement et yoksa obje üretemezsin" garantisi verir
2. **Type annotation** → Çalışma zamanında kural **koymaz**, sadece belgeleme ve IDE desteği sağlar
3. **Overloading** → Python'da `@overload` sadece tip bilgisi içindir; asıl mantık tek gerçek metodda yazılır
4. **`@override`** → Override ettiğini açıkça belirt; üst sınıf değişirse hemen haberdar olursun
5. **Decorator zincirleme** → İçten dışa uygulanır; sıra önemlidir

### Abstract Method

In [1]:
from abc import ABC, abstractmethod

In [2]:
class Animal(ABC):

    def __init__(self, name):
        self.name = name

    @abstractmethod
    def make_sound(self):
        pass

    @abstractmethod
    def move(self):
        pass

In [3]:
class Dog(Animal):

    def make_sound(self):
        return "hav hav"

    def move(self):
        return "walked"

In [4]:
dog = Dog("Karabaş")
print(f"{dog.name} {dog.move()}")
print(f"{dog.make_sound()} {dog.name}")

Karabaş walked
hav hav Karabaş


### overloading, overriding, final

In [5]:
def example(x: int): 
    """
    Type annotations Python’da zorunlu değildir, runtime’da tip kontrolü yapmaz. 
    Ancak kodun okunabilirliğini artırır ve IDE ile statik analiz araçlarının olası tip hatalarını önceden yakalamasını sağlar.
    """
    return x * 2

In [6]:
print(example(2))
print(example("A"))

4
AA


In [7]:
from typing import overload

class Calculator:

    def add(self, a: int, b: int) -> int:
        return a + b

    def add(self, a: int, b: int, c: int) -> int:
        return a + b + c

In [8]:
cal1 = Calculator()
# cal.add(10,20) -> TypeError: Calculator.add() missing 1 required positional argument: 'c'
# aynı sınıf içerisinde; aynı isimli, farklı parametreli metotlar yazarsanız en son yazılan geçerli olur
print(cal1.add(1,2,3))

6


In [9]:
class Calculator:

    @overload
    def add(self, a: int, b: int) -> int:
        ...

    @overload
    def add(self, a: int, b: int, c: int) -> int:
        ...

    def add(self, a: int, b:int, c:int | None = None) -> int:
        if c is None:
            return a + b
        return a + b + c

In [10]:
cal2 = Calculator()
print(cal2.add(10,20))
print(cal2.add(10,20,30))

30
60


In [11]:
class Calculator:

    # @overload kullanmasak da çalışır, Burada @overload runtime’da hiçbir şey yapmaz, sadece IDE ve type checker’a yardımcı olur
    
    def add(self, a: int, b:int, c:int | None = None) -> int:
        if c is None:
            return a + b
        return a + b + c

cal3 = Calculator()
print(cal3.add(10,20))
print(cal3.add(10,20,30))

30
60


In [12]:
from typing import overload, Union

class Process:
    # Type hints kodun davranışını değiştirmez; okunabilirliği ve takım çalışmasını kolaylaştırır

    @overload
    def process(self, value: int) -> int:
        ...

    @overload
    def process(self, value: str) -> str:
        ...

    def process(self, value: Union[int, str]) -> Union[int, str]:
        if isinstance(value, int):
            return value * 3
        elif isinstance (value, str):
            return value.upper()
        else:
            raise ValueError("Value must be a string or a integer")

In [13]:
pro = Process()
print(pro.process("ensar"))
print(pro.process(11))
# print(pro.process([10,20,30]))-> ValueError: Value must be a string or a integer

ENSAR
33


In [14]:
from typing import final 

class BaseGame:

    def start(self):
        print("game started")

    @final
    def calculate_score(self, points: int) -> int:
        bonus = 100
        return points + bonus

    def end(self):
        print("game over")


class MyGame(BaseGame):

    def start(self):
        # override
        print("my game started")

    def calculate_score(self, points: int) -> int:
        return points * 2

In [15]:
my_game = MyGame()
my_game.start()

my game started


In [16]:
# @final -> metodun override edilmesini engellemek için kullanılır
# Runtime’da yine de çalışır; IDE veya type checker uyarı verir
my_game.calculate_score(100)

200

In [17]:
from typing import override

class Shape:

    def area(self) -> float:
        return 0.0

    def perimeter(self) -> float:
        return 0.0
        
class Rectangle(Shape):

    # @override -> üst sınıftaki metodu doğru şekilde override ettiğimizi IDE/type checker’a bildirir

    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    @override
    def area(self) -> float:
        return self.width * self.height

    @override
    def perimeter(self) -> float:
        return 2 * (self.width + self.height)

In [18]:
shape1 = Rectangle(5, 4)
print(shape1.area())
print(shape1.perimeter())

20
18


### Bonus: Decorator Combining

In [19]:
def multiply_decorator(func):
    def wrapper(x: int):
       return func(x) * 2
    return wrapper
    
@multiply_decorator
def calculate(x: int):
    return x * 2

In [20]:
calculate(10)

40

In [21]:
def multiply_decorator(func):
    def wrapper(x: int):
       return func(x) * 2
    return wrapper

def other_decorator(func):
    def wrapper(x: int):
       return func(x) * 4
    return wrapper
    
@multiply_decorator
@other_decorator
def calculate(x: int):
    return x * 2

In [22]:
calculate(10)

160

In [23]:
# https://github.com/atilsamancioglu/P28-AdvancedPython/blob/main/python_advanced_tutorial.py